In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import re

df = pd.read_csv('IMDB Dataset.csv')

In [16]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    return text

df['clean_review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [17]:
embeddings = {}

with open('wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        try:
            vector = np.array(values[1:], dtype='float32')
            if len(vector) == 100:  # only store if exactly 100 dimensions
                embeddings[word] = vector
        except ValueError:
            continue  # skip bad lines silently

print(f"Loaded {len(embeddings)} word vectors")

Loaded 1287614 word vectors


In [18]:
def review_to_vector(review):
    words = review.split()
    vectors = [embeddings[word] for word in words if word in embeddings]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

df['vector'] = df['clean_review'].apply(review_to_vector)

In [19]:
# test a few words
print(embeddings['terrible'])
print("---")
print(embeddings['great'])

[ 0.266868  0.634706 -0.531134  0.338849 -0.438938 -0.485722 -0.378986
 -0.690481  0.362781  0.460933  1.15716   0.353177 -0.492698  0.105623
  0.46298  -0.010164 -0.696472 -0.713614 -0.484673  0.090101 -0.326005
 -0.240383  0.418077  0.573668 -0.309485  1.195817  0.142061 -0.879642
  0.413227 -0.063229 -0.545352  0.523708  0.46542   0.444556 -2.179207
 -0.032157  0.53138   0.160324  0.39026  -0.124863 -0.170558 -0.199414
  0.468345  0.029985  0.537022  1.005084 -0.22478   0.5233    1.373777
 -0.611404 -0.232244  0.353347  0.326262  0.187582 -0.090673  0.06653
  0.398911  0.110634 -0.388842  0.418699  0.475354  0.058279 -0.266079
  0.334101 -1.677997 -0.266109  0.413943 -0.033702 -0.338059  0.320278
 -0.105472 -0.195374 -0.682866 -0.187634  0.435552  0.156068 -1.038901
 -0.118925 -0.09718   0.356871  0.092192  0.524181 -0.207248  0.522713
 -0.661598  0.366663  0.714536  0.003942 -0.146909  0.016072  0.27533
 -0.814034 -1.001654 -0.172495  0.404072  0.3883   -0.095801 -0.253113
  0.4915

In [20]:
X = np.stack(df['vector'].values)
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

Accuracy: 0.7862
